In [32]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import fbeta_score, recall_score
from sklearn.utils import class_weight
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, GRU, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
import joblib

def find_best_thr_proba(y_true: np.ndarray,
                       y_proba: np.ndarray,
                       beta: float = 2.0,
                       grid: np.ndarray | None = None
                       ) -> tuple[float, float]:
    if grid is None:
        grid = np.linspace(0, 1, 101)
    best_thr, best_f2 = 0.5, -1.0
    for t in grid:
        f2 = fbeta_score(y_true, (y_proba >= t).astype(int), beta=beta)
        if f2 > best_f2:
            best_f2, best_thr = f2, t
    return best_thr, best_f2

def focal_loss(gamma=2.0, alpha=0.25):
    import tensorflow.keras.backend as K
    def loss(y_true, y_pred):
        y_pred = K.clip(y_pred, K.epsilon(), 1. - K.epsilon())
        pt = tf.where(K.equal(y_true, 1), y_pred, 1 - y_pred)
        w = alpha * K.pow(1. - pt, gamma)
        return K.mean(-w * K.log(pt))
    return loss

# 1) Carrega o CSV achatado
df = pd.read_csv(Path("../csv/train_seq_data.csv"))

EVENT_COL = "event_id"
TARGET    = "risk"

# 2) Monta X 3D
feat_cols = [c for c in df.columns if c not in (EVENT_COL, TARGET)]
lags = sorted({int(re.search(r"_t-(\d+)$", c).group(1)) for c in feat_cols})
TIMESTEPS = len(lags)
n_feats   = len(feat_cols) // TIMESTEPS

def sort_key(col):
    m = re.search(r"(.*)_t-(\d+)$", col)
    return (int(m.group(2)), m.group(1))

feat_cols_sorted = sorted(feat_cols, key=sort_key)
df_feats = (df[feat_cols_sorted]
              .replace({'False': 0, 'True': 1})
              .astype(float))
X_flat   = df_feats.values
n_events = X_flat.shape[0]
X        = X_flat.reshape(n_events, TIMESTEPS, n_feats)
y        = df[TARGET].values

print("X.shape =", X.shape)
print("y.shape =", y.shape)

# 3) Normalização
X2d    = X.reshape(-1, n_feats)
scaler = StandardScaler()
X_norm = scaler.fit_transform(X2d).reshape(n_events, TIMESTEPS, n_feats)

# 4) Split estratificado em high/low risk
thr      = -6.0
y_bin    = (y >= thr).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_norm, y, test_size=0.2, stratify=y_bin, random_state=42
)
y_tr_bin = (y_tr >= thr).astype(int)
y_te_bin = (y_te >= thr).astype(int)

# 5) SMOTE no treino
n_tr       = X_tr.shape[0]
X_tr_flat  = X_tr.reshape(n_tr, TIMESTEPS * n_feats)
sm         = SMOTE(random_state=42)
X_res_flat, y_res_bin = sm.fit_resample(X_tr_flat, y_tr_bin)
X_res      = X_res_flat.reshape(-1, TIMESTEPS, n_feats)

print("Após SMOTE → X_res.shape =", X_res.shape,
      "y_res_bin distrib:", np.bincount(y_res_bin))

# 6) Calcula class_weight (opcional)
cw = class_weight.compute_class_weight(
    'balanced', classes=np.unique(y_res_bin), y=y_res_bin
)
class_weight_dict = dict(enumerate(cw))
print("class_weight:", class_weight_dict)

# 7) Define modelo GRU
model = Sequential([
    Masking(mask_value=0., input_shape=(TIMESTEPS, n_feats)),
    GRU(64, return_sequences=False),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

use_focal = True  # ou False para bce
model.compile(
    optimizer='adam',
    loss= focal_loss(gamma=2.0, alpha=0.25) if use_focal else 'binary_crossentropy',
    metrics=[tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)
model.summary()

# 8) Treino com class_weight
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(
    X_res, y_res_bin,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[es],
    class_weight=class_weight_dict,
    verbose=1
)

# 9) Avaliação no teste
y_proba = model.predict(X_te).ravel()

# F2 @ 0.5
f2_at_05    = fbeta_score(y_te_bin, (y_proba >= 0.5).astype(int), beta=2.0)
recall_at_05 = recall_score(y_te_bin, (y_proba >= 0.5).astype(int))
print(f"F2 @0.5      = {f2_at_05:.4f}")
print(f"Recall @0.5  = {recall_at_05:.4f}")

# Melhor threshold
best_thr, best_f2 = find_best_thr_proba(y_te_bin, y_proba, beta=2.0)
recall_best     = recall_score(y_te_bin, (y_proba >= best_thr).astype(int))
print(f"Melhor thr   = {best_thr:.3f}  |  F2 = {best_f2:.4f}")
print(f"Recall @thr  = {recall_best:.4f}")

# 10) Salva artefatos
model.save("artifacts/gru_model_with_smote_classweight_focal.keras")
joblib.dump(scaler, "artifacts/seq_scaler.pkl")


X.shape = (13154, 5, 117)
y.shape = (13154,)
Após SMOTE → X_res.shape = (18880, 5, 117) y_res_bin distrib: [9440 9440]
class_weight: {0: 1.0, 1: 1.0}


c:\Python312\Lib\site-packages\keras\src\layers\core\masking.py:47: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_9 (Masking)             │ (None, 5, 117)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_9 (GRU)                     │ (None, 64)             │        35,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,201 (137.50 KB)

 Trainable params: 35,201 (137.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
472/472 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.0232 - precision: 0.7830 - recall: 0.8286 - val_loss: 0.0090 - val_precision: 1.0000 - val_recall: 0.9510
Epoch 2/100
472/472 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0065 - precision: 0.9530 - recall: 0.9585 - val_loss: 0.0047 - val_precision: 1.0000 - val_recall: 0.9759
Epoch 3/100
472/472 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0038 - precision: 0.9667 - recall: 0.9821 - val_loss: 0.0031 - val_precision: 1.0000 - val_recall: 0.9950
Epoch 4/100
472/472 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0027 - precision: 0.9774 - recall: 0.9871 - val_loss: 0.0014 - val_precision: 1.0000 - val_recall: 0.9989
Epoch 5/100
472/472 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0021 - precision: 0.9819 - recall: 0.9927 - val_loss: 7.7991e-04 - val_precision: 1.0000 - val_recall: 0.9995
Epoch 6/100
472/472 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0017 - precision: 0.9863 - recall: 0.9950 - val_loss: 7.8646e-04 - val_precision: 1.0000

['artifacts/seq_scaler.pkl']